# Notebook 6 -- Three-Angle Evaluation on SVAMP
## SLM-to-SLM Guided Reasoning Pipeline

**Why SVAMP instead of GSM8K?**

SVAMP (Simple Variations on Arithmetic Math Problems) is specifically designed
to break solvers that rely on surface patterns rather than real understanding.
The dataset takes GSM8K-style problems and applies small structural changes
(swapping what is asked, changing relations) -- a model that "guesses" from
keywords will fail. This means the guidance gap should be *larger and more
meaningful* than on GSM8K.

| Property | GSM8K | SVAMP |
|---|---|---|
| Size (test) | 1,319 | 1,000 |
| Difficulty | Grade school | Robustness-focused |
| Pattern-gaming risk | High | Low (by design) |
| HuggingFace ID | `openai/gsm8k` | `ChilleD/SVAMP` |

**Pipeline (unchanged from Notebook 5)**
```
Question --> Fine-tuned Qwen 3B (Guide) --> Plan --> Qwen 1.5B (Solver) x5 --> Vote --> Answer
Baseline : Question -----------------------------------------> Qwen 1.5B (Solver) x5 --> Vote
```

**Improvements in this notebook**
- Fixed random seed applied ONCE before sampling -- both modes use identical questions
- Richer answer extractor handles floats, LaTeX, bold markdown
- Per-question wasted-vote tracking (not just aggregate)
- Refiner effectiveness logged separately
- Angle 2 adds per-question win/loss/tie comparison
- Angle 3 adds explicit false-confidence count per bucket
- Summary table uses actual numeric cells (not f-string boxes)


In [3]:
# CELL 1 -- Install (uncomment on first run)
# !pip install -q transformers==4.44.0
# !pip install -q peft==0.12.0
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q huggingface_hub
print("Done.")


Done.


In [ ]:
# CELL 2 -- HuggingFace login
from huggingface_hub import login
login("")
print('✅ HuggingFace login done')


✅ HuggingFace login done


In [5]:
# CELL 3 -- Imports + GPU
import os, json, re, glob, random, time
import torch
import numpy as np
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
from tqdm.notebook import tqdm

OUTPUT_DIR = "/kaggle/working/svamp_eval"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Output  : {OUTPUT_DIR}")


PyTorch : 2.10.0+cu128
GPU     : Tesla T4
VRAM    : 15.6 GB
Output  : /kaggle/working/svamp_eval


In [6]:
# CELL 4 -- Configuration
CONFIG = {
    # Models
    "guide_base"          : "Qwen/Qwen2.5-3B-Instruct",
    "response_model"      : "Qwen/Qwen2.5-1.5B-Instruct",

    # Dataset
    "dataset_name"        : "ChilleD/SVAMP",   # huggingface dataset id
    "dataset_split"       : "test",
    "max_eval_samples"    : 500,   # SVAMP test has 1000; increase to 1000 for full run
    "random_seed"         : 42,    # FIXED -- same seed for BOTH guided and baseline

    # Ensemble
    "n_votes"             : 5,
    "vote_temperature"    : 0.7,
    "guide_temperature"   : 0.1,
    "refiner_temperature" : 0.3,
    "max_new_tokens"      : 350,

    # Compute cost (billions of parameters)
    "guide_params_B"      : 3.0,
    "solver_params_B"     : 1.5,

    # Paths
    "results_file"        : f"{OUTPUT_DIR}/results.jsonl",
    "report_file"         : f"{OUTPUT_DIR}/eval_report.json",
    "angle1_file"         : f"{OUTPUT_DIR}/angle1_compute_efficiency.json",
    "angle2_file"         : f"{OUTPUT_DIR}/angle2_vote_consistency.json",
    "angle3_file"         : f"{OUTPUT_DIR}/angle3_confidence_calibration.json",
    "checkpoint_file"     : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"          : 25,
}

print("Config ready:")
for k, v in CONFIG.items():
    print(f"  {k:<24}: {v}")


Config ready:
  guide_base              : Qwen/Qwen2.5-3B-Instruct
  response_model          : Qwen/Qwen2.5-1.5B-Instruct
  dataset_name            : ChilleD/SVAMP
  dataset_split           : test
  max_eval_samples        : 500
  random_seed             : 42
  n_votes                 : 5
  vote_temperature        : 0.7
  guide_temperature       : 0.1
  refiner_temperature     : 0.3
  max_new_tokens          : 350
  guide_params_B          : 3.0
  solver_params_B         : 1.5
  results_file            : /kaggle/working/svamp_eval/results.jsonl
  report_file             : /kaggle/working/svamp_eval/eval_report.json
  angle1_file             : /kaggle/working/svamp_eval/angle1_compute_efficiency.json
  angle2_file             : /kaggle/working/svamp_eval/angle2_vote_consistency.json
  angle3_file             : /kaggle/working/svamp_eval/angle3_confidence_calibration.json
  checkpoint_file         : /kaggle/working/svamp_eval/checkpoint.json
  save_every              : 25


In [7]:
# CELL 5 -- Load SVAMP dataset
# SVAMP fields: Body, Question, Equation, Answer (numeric)
# We combine Body + Question into a single question string.

print("Loading SVAMP from HuggingFace...")
raw_ds = load_dataset(CONFIG["dataset_name"])

print(f"Splits     : {list(raw_ds.keys())}")
print(f"Features   : {list(raw_ds[CONFIG['dataset_split']].features.keys())}")
print(f"Test size  : {len(raw_ds[CONFIG['dataset_split']])}")
print(f"Example    :")
ex = raw_ds[CONFIG["dataset_split"]][0]
for k, v in ex.items():
    print(f"  {k}: {v}")


def normalise_svamp(item):
    """Convert SVAMP record to {question, answer} used by the pipeline."""
    q = item["Body"].strip().rstrip(".") + " " + item["Question"].strip()
    ans = item["Answer"]
    # Store as clean integer string when possible (5.0 -> "5")
    if isinstance(ans, float) and ans == int(ans):
        ans_str = str(int(ans))
    else:
        ans_str = str(ans)
    return {"question": q, "answer": ans_str}


all_data = [normalise_svamp(x) for x in raw_ds[CONFIG["dataset_split"]]]

# ---- CRITICAL: set seed ONCE here, before any sampling ----------
random.seed(CONFIG["random_seed"])
if CONFIG["max_eval_samples"] < len(all_data):
    test_data = random.sample(all_data, CONFIG["max_eval_samples"])
    print(f"\nSampled {len(test_data)} questions (seed={CONFIG['random_seed']})")
else:
    test_data = all_data
    print(f"\nUsing all {len(test_data)} questions")

# Fingerprint so we can verify reproducibility
print(f"First Q  : {test_data[0]['question'][:80]}...")
print(f"First A  : {test_data[0]['answer']}")
print(f"Last  Q  : {test_data[-1]['question'][:60]}...")
print("SVAMP loaded")


Loading SVAMP from HuggingFace...


README.md:   0%|          | 0.00/675 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/111k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/54.8k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/700 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/300 [00:00<?, ? examples/s]

Splits     : ['train', 'test']
Features   : ['ID', 'Body', 'Question', 'Equation', 'Answer', 'Type', 'question_concat']
Test size  : 300
Example    :
  ID: chal-736
  Body: Winter is almost here and most animals are migrating to warmer countries. There are 41 bird families living near the mountain. If 35 bird families flew away to asia and 62 bird families flew away to africa
  Question: How many more bird families flew away to africa than those that flew away to asia?
  Equation: ( 62.0 - 35.0 )
  Answer: 27
  Type: Subtraction
  question_concat: Winter is almost here and most animals are migrating to warmer countries. There are 41 bird families living near the mountain. If 35 bird families flew away to asia and 62 bird families flew away to africa How many more bird families flew away to africa than those that flew away to asia?

Using all 300 questions
First Q  : Winter is almost here and most animals are migrating to warmer countries. There ...
First A  : 27
Last  Q  : Jake has 13 

In [8]:
# CELL 6 -- Answer extraction (upgraded for SVAMP)
# SVAMP answers are clean integers or simple floats -- no dollar signs or commas.
# We still need to handle all Qwen output styles.

def normalise_num(s):
    """Convert numeric string to canonical form. 5.0 -> '5', 3.14 -> '3.14'."""
    s = s.replace(",", "").strip()
    try:
        f = float(s)
        return str(int(f)) if f == int(f) else str(round(f, 4))
    except ValueError:
        return s


def extract_gt_answer(answer_str):
    """SVAMP GT is already clean -- just normalise."""
    return normalise_num(str(answer_str))


def extract_pred_answer(text):
    """
    Multi-pattern extractor. Returns empty string on failure
    (never falls back to a random number from the text).

    Priority:
      1. #### N          -- standard format we request
      2. \\boxed{N}      -- Qwen's preferred LaTeX style
      3. 'the answer is' -- common phrasing
      4. '= N' at end of line
      5. **N** at end    -- bold markdown
      6. 'therefore N'   -- conclusion phrases
    """
    # 1
    m = re.search(r"####\s*(-?[\d\.]+)", text)
    if m: return normalise_num(m.group(1))
    # 2
    m = re.search(r"\\boxed\{(-?[\d\.]+)\}", text)
    if m: return normalise_num(m.group(1))
    # 3
    m = re.search(r"(?:the answer is|answer is)\s*:?\s*\$?(-?[\d\.]+)", text, re.IGNORECASE)
    if m: return normalise_num(m.group(1))
    # 4
    m = re.search(r"=\s*\$?(-?[\d\.]+)\s*$", text.strip(), re.MULTILINE)
    if m: return normalise_num(m.group(1))
    # 5
    m = re.search(r"\*\*\$?(-?[\d\.]+)\*\*\.?\s*$", text.strip())
    if m: return normalise_num(m.group(1))
    # 6
    m = re.search(r"(?:therefore|thus|so|hence)[,\s]+(?:the answer is\s*)?\$?(-?[\d\.]+)", text, re.IGNORECASE)
    if m: return normalise_num(m.group(1))
    return ""


# --- Self-test ---
_tests = [
    ("#### 42",                 "42"),
    ("#### 3.5",                "3.5"),
    ("\\boxed{100}",            "100"),
    ("The answer is 7",         "7"),
    ("Total = 20",              "20"),
    ("**200**.",                "200"),
    ("Therefore, 13",           "13"),
    ("Some unrelated text",     ""),
]
ok = True
for txt, exp in _tests:
    got = extract_pred_answer(txt)
    status = "OK" if got == exp else "FAIL"
    if got != exp: ok = False
    print(f"  {status}  '{txt[:35]}' -> '{got}' (expected '{exp}')")
print("All extractor tests passed" if ok else "EXTRACTOR HAS FAILURES -- fix before running eval")


  OK  '#### 42' -> '42' (expected '42')
  OK  '#### 3.5' -> '3.5' (expected '3.5')
  OK  '\boxed{100}' -> '100' (expected '100')
  OK  'The answer is 7' -> '7' (expected '7')
  OK  'Total = 20' -> '20' (expected '20')
  OK  '**200**.' -> '200' (expected '200')
  OK  'Therefore, 13' -> '13' (expected '13')
  OK  'Some unrelated text' -> '' (expected '')
All extractor tests passed


In [9]:
# CELL 7 -- Load fine-tuned guide model (Qwen 3B + LoRA)

def find_adapter():
    patterns = [
        "/kaggle/input/datasets/sufiantabdullah/final-adapter-asdiv",
        "/kaggle/input/datasets/sufiantabdullah/final-adapter",
        "/kaggle/input/*/adapter",
        "/kaggle/input/*/final-adapter",
        "/kaggle/input/*/final_adapter",
    ]
    for p in patterns:
        for m in glob.glob(p):
            print(f"  Found adapter: {m}")
            return m
    return None


print(f"Loading guide base: {CONFIG['guide_base']}")
guide_tok = AutoTokenizer.from_pretrained(CONFIG["guide_base"], trust_remote_code=True)
guide_tok.padding_side = "left"
if guide_tok.pad_token is None:
    guide_tok.pad_token = guide_tok.eos_token

guide_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["guide_base"],
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)

adapter_path = find_adapter()
if adapter_path:
    guide_model = PeftModel.from_pretrained(guide_model, adapter_path)
    print("LoRA adapter loaded -- fine-tuned guide active")
else:
    print("WARNING: No adapter found. Using base Qwen 3B as guide.")
    print("Results will still run but guide quality may be lower.")

guide_model.eval()
print(f"Guide VRAM : {torch.cuda.memory_allocated() / 1e9:.2f} GB")


Loading guide base: Qwen/Qwen2.5-3B-Instruct


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  Found adapter: /kaggle/input/datasets/sufiantabdullah/final-adapter-asdiv
LoRA adapter loaded -- fine-tuned guide active
Guide VRAM : 3.14 GB


In [10]:
# CELL 8 -- Load solver model (Qwen 1.5B)

print(f"Loading solver: {CONFIG['response_model']}")
resp_tok = AutoTokenizer.from_pretrained(CONFIG["response_model"])
if resp_tok.pad_token is None:
    resp_tok.pad_token = resp_tok.eos_token

resp_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["response_model"],
    torch_dtype=torch.float16,
    device_map="auto",
).eval()

total_vram = torch.cuda.memory_allocated() / 1e9
headroom   = 17.1 - total_vram
print(f"Total VRAM (both models) : {total_vram:.2f} GB / 17.1 GB")
print(f"Headroom                 : {headroom:.1f} GB")
if headroom < 2:
    print("WARNING: Very tight. Reduce n_votes to 3 if you see OOM errors.")
else:
    print("Memory OK")


Loading solver: Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Total VRAM (both models) : 4.83 GB / 17.1 GB
Headroom                 : 12.3 GB
Memory OK


In [11]:
# CELL 9 -- Prompts and generation functions

GUIDE_SYSTEM = (
    "You are a math problem decomposition assistant.\n"
    "Identify ONLY the arithmetic operations needed. 1-3 steps maximum.\n"
    "State WHAT is being compared or combined using EXACT numbers.\n"
    "Do NOT invent steps. Do NOT reorder the question.\n\n"
    "BAD:  Step 1: Calculate remaining cookies.\n"
    "GOOD: Step 1: Eaten - Given = 14 - 13 = ?\n"
    "      Step 2: Answer = 14 - 13\n\n"
    "If the question asks 'how many MORE did X than Y', "
    "the operation is X - Y, not Y - X.\n"
    "No final answer number. Just the operation steps."
)

SOLVE_SYSTEM = (
    "You are a math problem solver.\n"
    "Compute each step numerically. No markdown. No bullet points. No headers.\n"
    "Write plain arithmetic steps only.\n"
    "Your absolute last line must be: #### [number]\n"
    "NEVER write ### or ** in your response.\n\n"
    "Example:\n"
    "Eaten = 14. Given = 13.\n"
    "Difference = 14 - 13 = 1.\n"
    "#### 1"
)

BASELINE_SYSTEM = (
    "You are a precise math problem solver.\n"
    "Read the problem carefully. Solve step by step, showing every calculation.\n"
    "Your FINAL line must be exactly: #### [number]"
)

REFINER_SYSTEM = (
    "You are a careful math problem solver.\n"
    "Previous attempts on this problem gave different answers.\n"
    "Re-solve completely from scratch using a fresh approach.\n"
    "Show every arithmetic step.\n"
    "Your FINAL line must be exactly: #### [number]"
)


def run_qwen(mdl, tok, messages, max_tokens, temperature):
    """Call any Qwen-family model and return generated text."""
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt", truncation=True, max_length=1024)
    device = next(mdl.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens     = max_tokens,
            temperature        = max(temperature, 0.05),
            do_sample          = True,
            top_p              = 0.92,
            top_k              = 40,
            pad_token_id       = tok.eos_token_id,
            repetition_penalty = 1.15,
        )
    new_toks = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(new_toks, skip_special_tokens=True).strip()


def generate_plan(question):
    return run_qwen(
        guide_model, guide_tok,
        [{"role": "system", "content": GUIDE_SYSTEM},
         {"role": "user",   "content": f"Problem: {question}"}],
        max_tokens  = 350,
        temperature = CONFIG["guide_temperature"],
    )


def generate_guided(question, plan):
    content = f"Problem: {question}\n\nPlan (follow each step):\n{plan}\n\nSolve step by step:"
    return run_qwen(
        resp_model, resp_tok,
        [{"role": "system", "content": SOLVE_SYSTEM},
         {"role": "user",   "content": content}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["vote_temperature"],
    )


def generate_baseline(question):
    return run_qwen(
        resp_model, resp_tok,
        [{"role": "system", "content": BASELINE_SYSTEM},
         {"role": "user",   "content": f"Problem: {question}"}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["vote_temperature"],
    )


def generate_refiner(question, plan, candidates):
    cands = ", ".join(sorted(set(c for c in candidates if c)))
    content = (
        f"Problem: {question}\n\n"
        f"Plan:\n{plan}\n\n"
        f"Previous attempts disagreed: {cands}\n"
        "Re-solve carefully from scratch:"
    )
    return run_qwen(
        resp_model, resp_tok,
        [{"role": "system", "content": REFINER_SYSTEM},
         {"role": "user",   "content": content}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["refiner_temperature"],
    )


print("Generation functions ready")


Generation functions ready


In [12]:
# CELL 10 -- Voting logic with richer metrics

def vote_and_decide(answers, question, plan, gt_answer=None):
    """
    Majority voting with refiner fallback on ties.
    Returns a dict with all metrics needed for the three angles.

    Fields:
      final_answer     : chosen answer string
      strategy         : 'majority' | 'refiner_tiebreak' | 'coin_flip'
      confidence       : top_count / total_votes
      correct_votes    : votes that matched gt_answer
      vote_consistency : correct_votes / total_votes
      wasted_votes     : votes that did NOT match final_answer
      refiner_used     : bool
      refiner_correct  : bool or None
    """
    valid = [a for a in answers if a and a.strip()]
    if not valid:
        valid = answers  # fallback — keep all if everything failed
    vote_counts = Counter(valid)
    most_common = vote_counts.most_common()
    top_answer  = most_common[0][0]
    top_count   = most_common[0][1]
    total       = len(answers)

    correct_votes    = vote_counts.get(gt_answer, 0) if gt_answer else 0
    vote_consistency = correct_votes / total

    is_majority = (len(most_common) == 1 or top_count > most_common[1][1])

    refiner_used    = False
    refiner_correct = None

    if is_majority:
        final    = top_answer
        strategy = "majority"
        conf     = round(top_count / total, 4)
        wasted   = total - top_count
    else:
        # Tie: call refiner as an extra vote
        ref_raw    = generate_refiner(question, plan, list(answers))
        ref_ans    = extract_pred_answer(ref_raw)
        refiner_used    = True
        refiner_correct = (ref_ans == gt_answer) if gt_answer else None

        all_votes  = answers + [ref_ans]
        new_counts = Counter(all_votes)
        new_common = new_counts.most_common()
        new_top    = new_common[0][0]
        new_top_c  = new_common[0][1]
        still_tied = len(new_common) > 1 and new_top_c == new_common[1][1]

        final      = new_top
        strategy   = "coin_flip" if still_tied else "refiner_tiebreak"
        conf       = round(new_top_c / len(all_votes), 4)
        vote_counts = new_counts
        total      = len(all_votes)
        correct_votes    = new_counts.get(gt_answer, 0) if gt_answer else 0
        vote_consistency = correct_votes / total
        wasted           = total - new_top_c

    return {
        "final_answer"     : final,
        "strategy"         : strategy,
        "confidence"       : conf,
        "vote_counts"      : dict(vote_counts),
        "correct_votes"    : correct_votes,
        "total_votes"      : total,
        "vote_consistency" : round(vote_consistency, 4),
        "wasted_votes"     : wasted,
        "refiner_used"     : refiner_used,
        "refiner_correct"  : refiner_correct,
    }


print("Voting logic ready")
print("  majority        -> clear winner across votes")
print("  refiner_tiebreak-> tie broken by refiner call")
print("  coin_flip       -> still tied after refiner")


Voting logic ready
  majority        -> clear winner across votes
  refiner_tiebreak-> tie broken by refiner call
  coin_flip       -> still tied after refiner


In [13]:
# CELL 11 -- Single question test (verify pipeline end-to-end)

print("=" * 65)
print("SINGLE QUESTION TEST  (SVAMP)")
print("=" * 65)

item = test_data[0]
q    = item["question"]
gt   = extract_gt_answer(item["answer"])
print(f"Question : {q}")
print(f"GT Answer: {gt}")

# Guided
print("\n[1] Guide generating plan...")
plan = generate_plan(q)
print(f"Plan:\n{plan}")

print(f"\n[2] Guided votes ({CONFIG['n_votes']}x)...")
guided_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_guided(q, plan)
    pred = extract_pred_answer(raw)
    guided_votes.append(pred)
    print(f"  Vote {i+1}: '{pred}'  |  raw[:80]: {raw[:80]}")

g = vote_and_decide(guided_votes, q, plan, gt)
print(f"\n  Result   : {g['final_answer']}  (GT: {gt})  {'CORRECT' if g['final_answer']==gt else 'WRONG'}")
print(f"  Strategy : {g['strategy']}")
print(f"  Confidence: {g['confidence']}")
print(f"  Correct votes: {g['correct_votes']}/{g['total_votes']} ({g['vote_consistency']*100:.0f}%)")

# Baseline
print("\n[3] Baseline votes (no plan)...")
base_votes = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_baseline(q)
    pred = extract_pred_answer(raw)
    base_votes.append(pred)
    print(f"  Vote {i+1}: '{pred}'")

b = vote_and_decide(base_votes, q, "baseline", gt)
print(f"\n  Baseline result : {b['final_answer']}  (GT: {gt})  {'CORRECT' if b['final_answer']==gt else 'WRONG'}")
print("\nPipeline verified -- run Cell 12 for full evaluation")


SINGLE QUESTION TEST  (SVAMP)
Question : Winter is almost here and most animals are migrating to warmer countries. There are 41 bird families living near the mountain. If 35 bird families flew away to asia and 62 bird families flew away to africa How many more bird families flew away to africa than those that flew away to asia?
GT Answer: 27

[1] Guide generating plan...
Plan:
Step 1: Families flying to Africa - Families flying to Asia = ?

[2] Guided votes (5x)...
  Vote 1: ''  |  raw[:80]: Families flying to Africa: 62
Families flying to Asia: 35

Calculation:

62 - 35
  Vote 2: '27'  |  raw[:80]: Families flying to Africa - Families flying to Asia = 62 - 35

Perform calculati
  Vote 3: '27'  |  raw[:80]: Families flying to Africa - Families flying to Asia = 62 - 35

First, subtract 3
  Vote 4: '27'  |  raw[:80]: Families flying to Africa - Families flying to Asia = 

First, let's find out ho
  Vote 5: '27'  |  raw[:80]: Families flying to Africa - Families flying to Asia = 

62 - 35

In [14]:
# CELL 12 -- Full Dual Evaluation Loop
#
# Runs every question TWICE with the SAME questions (seed fixed in Cell 5):
#   Mode A: Guided  (guide plan + solver x5)
#   Mode B: Baseline (solver x5, no plan)
#
# Both modes share the exact same question list.
# All per-question metrics are embedded directly in each result record.

print(f"Dual evaluation: {len(test_data)} SVAMP questions")
print(f"Each question: {CONFIG['n_votes']} guided votes + {CONFIG['n_votes']} baseline votes")
print("-" * 65)

all_results  = []   # guided results
base_results = []   # baseline results
start_idx    = 0

# Resume from checkpoint
if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            lines = [json.loads(l) for l in f if l.strip()]
        all_results  = [r for r in lines if r.get("mode") == "guided"]
        base_results = [r for r in lines if r.get("mode") == "baseline"]
    print(f"Resumed from index {start_idx}")
    print(f"  Guided saved: {len(all_results)}  Baseline saved: {len(base_results)}")
else:
    print("Starting fresh")

t0 = time.time()

for idx in tqdm(range(start_idx, len(test_data)), desc="SVAMP Eval"):
    item      = test_data[idx]
    question  = item["question"]
    gt_answer = extract_gt_answer(item["answer"])

    # ---- GUIDED -------------------------------------------------
    try:
        plan        = generate_plan(question)
        g_votes_raw = [extract_pred_answer(generate_guided(question, plan))
                       for _ in range(CONFIG["n_votes"])]
        g_dec       = vote_and_decide(g_votes_raw, question, plan, gt_answer)

        all_results.append({
            "mode"             : "guided",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "final_answer"     : g_dec["final_answer"],
            "correct"          : g_dec["final_answer"] == gt_answer,
            "strategy"         : g_dec["strategy"],
            "confidence"       : g_dec["confidence"],
            "correct_votes"    : g_dec["correct_votes"],
            "total_votes"      : g_dec["total_votes"],
            "vote_consistency" : g_dec["vote_consistency"],
            "wasted_votes"     : g_dec["wasted_votes"],
            "refiner_used"     : g_dec["refiner_used"],
            "refiner_correct"  : g_dec["refiner_correct"],
            "vote_counts"      : g_dec["vote_counts"],
            "plan"             : plan,
        })
    except RuntimeError as e:
        all_results.append({
            "mode": "guided", "idx": idx, "question": question,
            "gt_answer": gt_answer, "final_answer": "", "correct": False,
            "strategy": "error", "confidence": 0.0,
            "correct_votes": 0, "total_votes": CONFIG["n_votes"],
            "vote_consistency": 0.0, "wasted_votes": CONFIG["n_votes"],
            "refiner_used": False, "refiner_correct": None,
            "vote_counts": {}, "error": str(e),
        })

    # ---- BASELINE -----------------------------------------------
    try:
        b_votes_raw = [extract_pred_answer(generate_baseline(question))
                       for _ in range(CONFIG["n_votes"])]
        b_dec       = vote_and_decide(b_votes_raw, question, "baseline", gt_answer)

        base_results.append({
            "mode"             : "baseline",
            "idx"              : idx,
            "question"         : question,
            "gt_answer"        : gt_answer,
            "final_answer"     : b_dec["final_answer"],
            "correct"          : b_dec["final_answer"] == gt_answer,
            "strategy"         : b_dec["strategy"],
            "confidence"       : b_dec["confidence"],
            "correct_votes"    : b_dec["correct_votes"],
            "total_votes"      : b_dec["total_votes"],
            "vote_consistency" : b_dec["vote_consistency"],
            "wasted_votes"     : b_dec["wasted_votes"],
            "refiner_used"     : b_dec["refiner_used"],
            "refiner_correct"  : None,
            "vote_counts"      : b_dec["vote_counts"],
        })
    except RuntimeError as e:
        base_results.append({
            "mode": "baseline", "idx": idx, "question": question,
            "gt_answer": gt_answer, "final_answer": "", "correct": False,
            "strategy": "error", "confidence": 0.0,
            "correct_votes": 0, "total_votes": CONFIG["n_votes"],
            "vote_consistency": 0.0, "wasted_votes": CONFIG["n_votes"],
            "refiner_used": False, "refiner_correct": None,
            "vote_counts": {}, "error": str(e),
        })

    # Checkpoint
    if (idx + 1) % CONFIG["save_every"] == 0:
        with open(CONFIG["results_file"], "w") as f:
            for r in all_results + base_results:
                f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx + 1}, f)
        g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
        b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100
        mins  = (time.time() - t0) / 60
        print(f"  [{idx+1:3d}] Guided: {g_acc:.1f}%  Baseline: {b_acc:.1f}%  ({mins:.1f} min)")

# Final save
with open(CONFIG["results_file"], "w") as f:
    for r in all_results + base_results:
        f.write(json.dumps(r) + "\n")

g_c = sum(r["correct"] for r in all_results)
b_c = sum(r["correct"] for r in base_results)
print(f"\nEvaluation complete.")
print(f"  Guided   : {g_c}/{len(all_results)} = {g_c/len(all_results)*100:.1f}%")
print(f"  Baseline : {b_c}/{len(base_results)} = {b_c/len(base_results)*100:.1f}%")
print(f"  Delta    : +{(g_c/len(all_results) - b_c/len(base_results))*100:.1f} percentage points")
       

Dual evaluation: 300 SVAMP questions
Each question: 5 guided votes + 5 baseline votes
-----------------------------------------------------------------
Starting fresh


SVAMP Eval:   0%|          | 0/300 [00:00<?, ?it/s]

  [ 25] Guided: 72.0%  Baseline: 52.0%  (28.9 min)
  [ 50] Guided: 66.0%  Baseline: 56.0%  (53.0 min)
  [ 75] Guided: 61.3%  Baseline: 50.7%  (85.5 min)
  [100] Guided: 54.0%  Baseline: 47.0%  (117.5 min)
  [125] Guided: 52.0%  Baseline: 44.0%  (148.3 min)
  [150] Guided: 52.0%  Baseline: 45.3%  (175.8 min)
  [175] Guided: 49.7%  Baseline: 44.6%  (206.2 min)
  [200] Guided: 49.5%  Baseline: 41.5%  (244.2 min)
  [225] Guided: 49.3%  Baseline: 40.0%  (279.9 min)
  [250] Guided: 48.4%  Baseline: 40.0%  (313.5 min)
  [275] Guided: 49.1%  Baseline: 40.7%  (343.6 min)
  [300] Guided: 49.3%  Baseline: 41.3%  (371.8 min)

Evaluation complete.
  Guided   : 148/300 = 49.3%
  Baseline : 124/300 = 41.3%
  Delta    : +8.0 percentage points


In [15]:
# CELL 13 -- ANGLE 1: COMPUTE EFFICIENCY
# =================================================================
# We compare three setups by accuracy and compute cost:
#   Baseline : 1.5B solver x5 votes              = 7.5B param-passes
#   Guided   : 3B guide x1 + 1.5B solver x5      = 10.5B param-passes
#   Upper    : 3B model x5 (hypothetical ceiling) = 15.0B param-passes
#
# Key question: does the guided setup earn more accuracy per
# extra param-pass it spends compared to the baseline?
# =================================================================

G = CONFIG["guide_params_B"]    # 3.0
S = CONFIG["solver_params_B"]   # 1.5
N = CONFIG["n_votes"]           # 5

guided_compute   = (G * 1) + (S * N)    # 3 + 7.5 = 10.5
baseline_compute = S * N                 # 7.5
upper_compute    = G * N                 # 15.0

g_acc = sum(r["correct"] for r in all_results)  / len(all_results)  * 100
b_acc = sum(r["correct"] for r in base_results) / len(base_results) * 100

# Accuracy-per-billion-param-passes
g_eff = g_acc / guided_compute
b_eff = b_acc / baseline_compute

# Wasted votes: votes that did not match final answer
g_wasted       = sum(r["wasted_votes"] for r in all_results)
b_wasted       = sum(r["wasted_votes"] for r in base_results)
total_possible = len(all_results) * N

# Refiner stats
ref_triggered  = sum(r["refiner_used"] for r in all_results)
ref_correct    = sum(1 for r in all_results
                     if r["refiner_used"] and r.get("refiner_correct"))

# Per-strategy accuracy
strategy_stats = {}
for r in all_results:
    s = r["strategy"]
    if s not in strategy_stats:
        strategy_stats[s] = {"n": 0, "correct": 0}
    strategy_stats[s]["n"] += 1
    if r["correct"]:
        strategy_stats[s]["correct"] += 1

print("=" * 65)
print("ANGLE 1 -- COMPUTE EFFICIENCY  (SVAMP)")
print("=" * 65)
print(f"\n  {'Setup':<32} | {'Compute':>10} | {'Accuracy':>9} | {'Acc/B':>7}")
print(f"  {'-'*32}-+-{'-'*10}-+-{'-'*9}-+-{'-'*7}")
print(f"  {'Baseline (1.5B x ' + str(N) + ')':<32} | {baseline_compute:>8.1f}B  | {b_acc:>8.1f}% | {b_eff:>6.3f}")
print(f"  {'Guided  (3B x1 + 1.5B x' + str(N) + ')':<32} | {guided_compute:>8.1f}B  | {g_acc:>8.1f}% | {g_eff:>6.3f}")
print(f"  {'Upper   (3B x ' + str(N) + ')':<32} | {upper_compute:>8.1f}B  | {'(ceiling)':>9} |")

savings_pct = (1 - guided_compute / upper_compute) * 100
acc_gain    = g_acc - b_acc
print(f"\n  Accuracy gain over baseline  : +{acc_gain:.1f} percentage points")
print(f"  Compute savings vs upper     : {savings_pct:.0f}% cheaper")
print(f"  Efficiency lift (acc/B)      : {g_eff:.3f} vs {b_eff:.3f} (guided)")

print(f"\n  Wasted votes (votes != final answer):")
print(f"    Guided   : {g_wasted} / {total_possible}  ({g_wasted/total_possible*100:.1f}%)")
print(f"    Baseline : {b_wasted} / {total_possible}  ({b_wasted/total_possible*100:.1f}%)")
print(f"    Saved    : {b_wasted - g_wasted} fewer wasted compute passes with guidance")

if ref_triggered > 0:
    print(f"\n  Refiner (tie-breaker) stats:")
    print(f"    Triggered : {ref_triggered} / {len(all_results)} questions")
    print(f"    Correct   : {ref_correct} / {ref_triggered}  ({ref_correct/ref_triggered*100:.1f}% of ties resolved correctly)")

print(f"\n  Decision strategy breakdown (guided):")
print(f"  {'Strategy':<22} | {'Count':>6} | {'Accuracy':>9}")
print(f"  {'-'*22}-+-{'-'*6}-+-{'-'*9}")
for s, v in sorted(strategy_stats.items(), key=lambda x: -x[1]["n"]):
    acc_s = v["correct"] / v["n"] * 100 if v["n"] else 0
    print(f"  {s:<22} | {v['n']:>6} | {acc_s:>8.1f}%")

angle1 = {
    "dataset"               : "SVAMP", 
    "n_questions"           : len(all_results),
    "guided_compute_B"      : guided_compute,
    "baseline_compute_B"    : baseline_compute,
    "upper_compute_B"       : upper_compute,
    "guided_accuracy"       : round(g_acc, 2),
    "baseline_accuracy"     : round(b_acc, 2),
    "accuracy_gain"         : round(acc_gain, 2),
    "compute_savings_pct"   : round(savings_pct, 1),
    "guided_efficiency"     : round(g_eff, 4),
    "baseline_efficiency"   : round(b_eff, 4),
    "guided_wasted_votes"   : g_wasted,
    "baseline_wasted_votes" : b_wasted,
    "wasted_votes_saved"    : b_wasted - g_wasted,
    "refiner_triggered"     : ref_triggered,
    "refiner_correct"       : ref_correct,
    "strategy_breakdown"    : strategy_stats,
}
with open(CONFIG["angle1_file"], "w") as f:
    json.dump(angle1, f, indent=2)
print(f"\nSaved -> {CONFIG['angle1_file']}")


ANGLE 1 -- COMPUTE EFFICIENCY  (SVAMP)

  Setup                            |    Compute |  Accuracy |   Acc/B
  ---------------------------------+------------+-----------+--------
  Baseline (1.5B x 5)              |      7.5B  |     41.3% |  5.511
  Guided  (3B x1 + 1.5B x5)        |     10.5B  |     49.3% |  4.698
  Upper   (3B x 5)                 |     15.0B  | (ceiling) |

  Accuracy gain over baseline  : +8.0 percentage points
  Compute savings vs upper     : 30% cheaper
  Efficiency lift (acc/B)      : 4.698 vs 5.511 (guided)

  Wasted votes (votes != final answer):
    Guided   : 704 / 1500  (46.9%)
    Baseline : 809 / 1500  (53.9%)
    Saved    : 105 fewer wasted compute passes with guidance

  Refiner (tie-breaker) stats:
    Triggered : 63 / 300 questions
    Correct   : 12 / 63  (19.0% of ties resolved correctly)

  Decision strategy breakdown (guided):
  Strategy               |  Count |  Accuracy
  -----------------------+--------+----------
  majority               |   

In [16]:
# CELL 14 -- ANGLE 2: VOTE CONSISTENCY
# =================================================================
# Vote consistency = fraction of votes (out of 5) that matched GT.
# A high-consistency question means the model reliably solves it.
# A low-consistency question means it got lucky on the final vote.
#
# If guidance works, we expect BOTH:
#   - Higher mean consistency (more votes correct per question)
#   - More questions in the "high" bucket (4-5 correct votes)
#
# Per-question comparison: on each question, does guided produce
# more correct votes than baseline? Win/Loss/Tie.
# =================================================================

g_cons = [r["vote_consistency"] for r in all_results]
b_cons = [r["vote_consistency"] for r in base_results]

g_mean = np.mean(g_cons)
b_mean = np.mean(b_cons)
lift   = g_mean / max(b_mean, 1e-6)

# Per-question: guided better / baseline better / tied
guided_wins   = sum(1 for g, b in zip(g_cons, b_cons) if g > b)
baseline_wins = sum(1 for g, b in zip(g_cons, b_cons) if b > g)
tied          = sum(1 for g, b in zip(g_cons, b_cons) if g == b)

# Distribution buckets
def bucket(scores):
    return {
        "all_wrong  (0%)": sum(1 for s in scores if s == 0.0),
        "low       (1-39%)": sum(1 for s in scores if 0.0 < s < 0.4),
        "medium  (40-79%)": sum(1 for s in scores if 0.4 <= s < 0.8),
        "high   (80-100%)": sum(1 for s in scores if s >= 0.8),
    }

g_dist = bucket(g_cons)
b_dist = bucket(b_cons)

# For questions where the pipeline was CORRECT, how consistent were the votes?
# (high consistency + correct = true reliable solving, not a lucky majority)
g_corr_cons = [r["vote_consistency"] for r in all_results  if r["correct"]]
b_corr_cons = [r["vote_consistency"] for r in base_results if r["correct"]]

print("=" * 65)
print("ANGLE 2 -- VOTE CONSISTENCY  (SVAMP)")
print("=" * 65)
print(f"\n  Mean correct-vote ratio (out of {CONFIG['n_votes']} votes per question):")
print(f"    Guided   : {g_mean*100:.1f}%  ({g_mean*CONFIG['n_votes']:.2f} votes correct on average)")
print(f"    Baseline : {b_mean*100:.1f}%  ({b_mean*CONFIG['n_votes']:.2f} votes correct on average)")
print(f"    Lift     : {lift:.2f}x  (guided produces {lift:.1f}x more correct votes per question)")

print(f"\n  Per-question comparison (same questions, both modes):")
print(f"    Guided beats baseline : {guided_wins} / {len(all_results)} questions")
print(f"    Baseline beats guided : {baseline_wins} / {len(all_results)} questions")
print(f"    Equal                 : {tied} / {len(all_results)} questions")

print(f"\n  Distribution of vote consistency:")
print(f"  {'Bucket':<22} | {'Guided':>8} | {'Baseline':>8} | {'Diff':>6}")
print(f"  {'-'*22}-+-{'-'*8}-+-{'-'*8}-+-{'-'*6}")
for bkt in ["all_wrong  (0%)", "low       (1-39%)", "medium  (40-79%)", "high   (80-100%)"]:
    gv  = g_dist[bkt]
    bv  = b_dist[bkt]
    dif = gv - bv
    sign = "+" if dif >= 0 else ""
    print(f"  {bkt:<22} | {gv:>8} | {bv:>8} | {sign+str(dif):>6}")

if g_corr_cons:
    print(f"\n  Among CORRECT questions only -- average vote consistency:")
    print(f"    Guided   : {np.mean(g_corr_cons)*100:.1f}%  (n={len(g_corr_cons)})")
    print(f"    Baseline : {np.mean(b_corr_cons)*100:.1f}%  (n={len(b_corr_cons)})")
    print("    (High consistency + correct = genuine reliable solving, not lucky vote)")

angle2 = {
    "dataset"                   : "SVAMP",
    "n_questions"               : len(all_results),
    "guided_mean_consistency"   : round(g_mean, 4),
    "baseline_mean_consistency" : round(b_mean, 4),
    "consistency_lift"          : round(lift, 4),
    "guided_wins"               : guided_wins,
    "baseline_wins"             : baseline_wins,
    "tied"                      : tied,
    "guided_distribution"       : g_dist,
    "baseline_distribution"     : b_dist,
    "guided_correct_q_consistency"   : round(np.mean(g_corr_cons), 4) if g_corr_cons else 0,
    "baseline_correct_q_consistency" : round(np.mean(b_corr_cons), 4) if b_corr_cons else 0,
}
with open(CONFIG["angle2_file"], "w") as f:
    json.dump(angle2, f, indent=2)
print(f"\nSaved -> {CONFIG['angle2_file']}")


ANGLE 2 -- VOTE CONSISTENCY  (SVAMP)

  Mean correct-vote ratio (out of 5 votes per question):
    Guided   : 36.0%  (1.80 votes correct on average)
    Baseline : 28.1%  (1.41 votes correct on average)
    Lift     : 1.28x  (guided produces 1.3x more correct votes per question)

  Per-question comparison (same questions, both modes):
    Guided beats baseline : 149 / 300 questions
    Baseline beats guided : 91 / 300 questions
    Equal                 : 60 / 300 questions

  Distribution of vote consistency:
  Bucket                 |   Guided | Baseline |   Diff
  -----------------------+----------+----------+-------
  all_wrong  (0%)        |       79 |       84 |     -5
  low       (1-39%)      |       75 |      107 |    -32
  medium  (40-79%)       |       94 |       85 |     +9
  high   (80-100%)       |       52 |       24 |    +28

  Among CORRECT questions only -- average vote consistency:
    Guided   : 61.1%  (n=148)
    Baseline : 51.9%  (n=124)
    (High consistency + cor

In [17]:
# CELL 15 -- ANGLE 3: CONFIDENCE CALIBRATION
# =================================================================
# Confidence = fraction of votes that agreed on the winning answer.
# Perfect calibration: "80% confident" means 80% accurate.
#
# ECE (Expected Calibration Error) measures the average gap
# between stated confidence and actual accuracy across all buckets.
# LOWER ECE = more trustworthy confidence signal.
#
# False confidence = all 5 votes agree on the WRONG answer.
# This is the most dangerous failure: the system is maximally
# confident and maximally wrong simultaneously.
#
# Guidance should suppress false confidence by steering votes
# toward correct reasoning paths rather than shared mistakes.
# =================================================================

def calibration_report(results, label):
    buckets = [
        ("Very High  (>=0.80)", lambda c: c >= 0.80, 0.90),
        ("High       (0.60-0.80)", lambda c: 0.60 <= c < 0.80, 0.70),
        ("Medium     (0.40-0.60)", lambda c: 0.40 <= c < 0.60, 0.50),
        ("Low        (<0.40)",  lambda c: c < 0.40,  0.25),
    ]
    n_total   = len(results)
    ece       = 0.0
    calib_out = []

    false_conf = sum(1 for r in results if r["confidence"] >= 0.80 and not r["correct"])

    print(f"\n  [{label}]")
    print(f"  {'Confidence':<26} | {'N':>5} | {'Accuracy':>9} | {'Expected':>9} | {'Gap':>6} | Cal?")
    print(f"  {'-'*26}-+-{'-'*5}-+-{'-'*9}-+-{'-'*9}-+-{'-'*6}-+----")

    for name, cond, mid in buckets:
        subset = [r for r in results if cond(r["confidence"])]
        if not subset:
            print(f"  {name:<26} | {'--':>5} | {'--':>9} | {mid*100:>8.0f}% | {'--':>6} |")
            continue
        n   = len(subset)
        acc = sum(r["correct"] for r in subset) / n
        gap = abs(acc - mid)
        ece += (n / n_total) * gap
        flag = "Good" if gap < 0.15 else "Poor"
        print(f"  {name:<26} | {n:>5} | {acc*100:>8.1f}% | {mid*100:>8.0f}% | {gap:>6.3f} | {flag}")
        calib_out.append({
            "bucket": name, "count": n,
            "accuracy": round(acc, 4), "expected": mid, "gap": round(gap, 4)
        })

    hc_items = [r for r in results if r["confidence"] >= 0.80]
    hc_acc   = sum(r["correct"] for r in hc_items) / max(1, len(hc_items)) * 100
    print(f"  {'ECE (lower=better)':<26}   {ece:>5.4f}")
    print(f"  High-conf questions : {len(hc_items)}  |  Accuracy when confident: {hc_acc:.1f}%")
    print(f"  Confidently WRONG   : {false_conf} questions (false confidence)")
    return ece, calib_out, false_conf


print("=" * 65)
print("ANGLE 3 -- CONFIDENCE CALIBRATION  (SVAMP)")
print("=" * 65)
print("Ideal: accuracy at each confidence level matches that level.")
print("False confidence: model agrees on wrong answer with full certainty.")

g_ece, g_calib, g_false = calibration_report(all_results,  "GUIDED pipeline")
b_ece, b_calib, b_false = calibration_report(base_results, "BASELINE (no plan)")

improve_pct = (b_ece - g_ece) / max(b_ece, 1e-6) * 100
print(f"\n  ECE Summary:")
print(f"    Guided   ECE : {g_ece:.4f}")
print(f"    Baseline ECE : {b_ece:.4f}")
print(f"    Improvement  : {improve_pct:.1f}% better calibrated")
print(f"\n  False Confidence (confident AND wrong):")
print(f"    Guided   : {g_false} questions")
print(f"    Baseline : {b_false} questions")
print(f"    Reduction: {b_false - g_false} fewer false-confidence questions with guidance")

angle3 = {
    "dataset"                 : "SVAMP",
    "n_questions"             : len(all_results),
    "guided_ece"              : round(g_ece, 4),
    "baseline_ece"            : round(b_ece, 4),
    "ece_improvement_pct"     : round(improve_pct, 2),
    "guided_false_confidence" : g_false,
    "baseline_false_confidence": b_false,
    "false_conf_reduction"    : b_false - g_false,
    "guided_calibration"      : g_calib,
    "baseline_calibration"    : b_calib,
}
with open(CONFIG["angle3_file"], "w") as f:
    json.dump(angle3, f, indent=2)
print(f"\nSaved -> {CONFIG['angle3_file']}")


ANGLE 3 -- CONFIDENCE CALIBRATION  (SVAMP)
Ideal: accuracy at each confidence level matches that level.
False confidence: model agrees on wrong answer with full certainty.

  [GUIDED pipeline]
  Confidence                 |     N |  Accuracy |  Expected |    Gap | Cal?
  ---------------------------+-------+-----------+-----------+--------+----
  Very High  (>=0.80)        |    70 |     74.3% |       90% |  0.157 | Poor
  High       (0.60-0.80)     |    91 |     51.6% |       70% |  0.184 | Poor
  Medium     (0.40-0.60)     |    91 |     41.8% |       50% |  0.082 | Good
  Low        (<0.40)         |    48 |     22.9% |       25% |  0.021 | Good
  ECE (lower=better)           0.1207
  High-conf questions : 70  |  Accuracy when confident: 74.3%
  Confidently WRONG   : 18 questions (false confidence)

  [BASELINE (no plan)]
  Confidence                 |     N |  Accuracy |  Expected |    Gap | Cal?
  ---------------------------+-------+-----------+-----------+--------+----
  Very High  

In [18]:
# CELL 16 -- Full Paper Summary (all three angles)

with open(CONFIG["angle1_file"]) as f: a1 = json.load(f)
with open(CONFIG["angle2_file"]) as f: a2 = json.load(f)
with open(CONFIG["angle3_file"]) as f: a3 = json.load(f)

n  = a1["n_questions"]
ga = a1["guided_accuracy"]
ba = a1["baseline_accuracy"]
gc = a1["guided_compute_B"]
uc = a1["upper_compute_B"]

print("=" * 68)
print("  SVAMP EVALUATION -- PAPER SUMMARY TABLE")
print("=" * 68)
print(f"  Dataset  : SVAMP  |  Questions: {n}  |  Seed: {CONFIG['random_seed']}")
print(f"  Models   : Qwen 2.5-3B guide + Qwen 2.5-1.5B solver")
print()

rows = [
    ["Metric",                    "Baseline",       "Guided",          "Change"],
    ["Overall Accuracy",
     str(ba) + "%",               str(ga) + "%",
     "+" + str(round(ga-ba,1)) + " pts"],
    ["Compute Cost",
     str(a1['baseline_compute_B']) + "B param-passes",
     str(gc) + "B param-passes",
     str(a1['compute_savings_pct']) + "% cheaper than ceiling"],
    ["Wasted Votes",
     str(a1['baseline_wasted_votes']),
     str(a1['guided_wasted_votes']),
     str(a1['wasted_votes_saved']) + " fewer wasted passes"],
    ["Vote Consistency",
     str(round(a2['baseline_mean_consistency']*100,1)) + "%",
     str(round(a2['guided_mean_consistency']*100,1)) + "%",
     str(round(a2['consistency_lift'],2)) + "x lift"],
    ["High-Agreement Questions",
     str(a2['baseline_distribution']['high   (80-100%)']),
     str(a2['guided_distribution']['high   (80-100%)']),
     ""],
    ["Guided Wins Per-Question",
     "--",
     str(a2['guided_wins']) + " / " + str(n),
     ""],
    ["ECE (lower = better)",
     str(a3['baseline_ece']),
     str(a3['guided_ece']),
     str(a3['ece_improvement_pct']) + "% better"],
    ["False Confidence Count",
     str(a3['baseline_false_confidence']),
     str(a3['guided_false_confidence']),
     str(a3['false_conf_reduction']) + " fewer"],
]

col_w = [28, 18, 18, 28]
sep   = "-+-".join("-" * w for w in col_w)
for i, row in enumerate(rows):
    line = " | ".join(str(cell).ljust(col_w[j]) for j, cell in enumerate(row))
    print("  " + line)
    if i == 0:
        print("  " + sep)

if a1.get("refiner_triggered", 0) > 0:
    rt = a1["refiner_triggered"]
    rc = a1.get("refiner_correct", 0)
    print(f"\n  Refiner: triggered {rt} times, resolved {rc} correctly ({rc/rt*100:.1f}%)")

# Save full report
full = {
    "dataset": "SVAMP", "seed": CONFIG["random_seed"],
    "n_questions": n, "angle1": a1, "angle2": a2, "angle3": a3,
}
with open(CONFIG["report_file"], "w") as f:
    json.dump(full, f, indent=2)

print(f"\nAll results saved to {OUTPUT_DIR}")
print("Commit this notebook to preserve outputs.")


  SVAMP EVALUATION -- PAPER SUMMARY TABLE
  Dataset  : SVAMP  |  Questions: 300  |  Seed: 42
  Models   : Qwen 2.5-3B guide + Qwen 2.5-1.5B solver

  Metric                       | Baseline           | Guided             | Change                      
  -----------------------------+--------------------+--------------------+-----------------------------
  Overall Accuracy             | 41.33%             | 49.33%             | +8.0 pts                    
  Compute Cost                 | 7.5B param-passes  | 10.5B param-passes | 30.0% cheaper than ceiling  
  Wasted Votes                 | 809                | 704                | 105 fewer wasted passes     
  Vote Consistency             | 28.1%              | 36.0%              | 1.28x lift                  
  High-Agreement Questions     | 24                 | 52                 |                             
  Guided Wins Per-Question     | --                 | 149 / 300          |                             
  ECE (lower = bette